In [101]:
import sys
import os
import numpy as np

#Add the 'oracle' directory to the Python path
sys.path.append(os.path.join(os.getcwd(), 'oracle'))
import oracle
data = oracle.q3_hyper(23746)
print(data[0])
print(data[1])
print(data[2])

entropy
best
8


In [3]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Add column name in he first row of the csv file "processed.cleveland.data"
import csv

def add_column_names(file_path="processed.cleveland.data", output_file="processed_cleveland_with_headers.csv"):
    # Define column names based on the dataset
    column_names = [
        "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
        "thalach", "exang", "oldpeak", "slope", "ca", "thal", "goal"
    ]
    
    # Read the original data
    with open(file_path, "r") as infile:
        lines = infile.readlines()

    # Write the new file with headers
    with open(output_file, "w", newline="") as outfile:
        writer = csv.writer(outfile)
        writer.writerow(column_names)  # Write column headers
        for line in lines:
            writer.writerow(line.strip().split(","))  # Write original data rows

    print(f"File saved as {output_file} with column names added.")

# Call the function
add_column_names()


# Load dataset and convert '?' to NaN
df = pd.read_csv("processed_cleveland_with_headers.csv", na_values=["?"])  

# Convert entire DataFrame to numeric (forcing non-numeric to NaN)
df = df.apply(pd.to_numeric, errors='coerce')

# Check missing values before imputation
print("Missing values before imputation:\n", df.isna().sum())

# **Impute Numerical Columns with Median**
# what are all the strategies for SimpleImputer?
# https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html
num_imputer = SimpleImputer(strategy="mean")
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'exang', 'fbs', 'restecg', 'cp']
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# **Impute Categorical Columns with Mode**
cat_imputer = SimpleImputer(strategy="most_frequent")
cat_cols = ['slope', 'ca', 'thal']
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# Verify if there are any NaNs left
print("\nMissing values after imputation:\n", df.isna().sum())

# Print cleaned data
print("\nCleaned Data:\n", df.head())



# save a new file with the new data
df.to_csv("cleaned_data.csv", index=False)

File saved as processed_cleveland_with_headers.csv with column names added.
Missing values before imputation:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
goal        0
dtype: int64

Missing values after imputation:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
goal        0
dtype: int64

Cleaned Data:
     age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0 

In [ ]:
import sklearn
# we want to use decsion tree classifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# load the data
df = pd.read_csv("cleaned_data.csv")
features =[]
goals = []
import csv
f = open('cleaned_data.csv', 'r')
reader = csv.reader(f)
for row in reader:
    features.append(list(row[:-1]))
    goals.append(row[-1])
f.close()

features = features[1:]
goals = goals[1:]
for i in range(len(goals)) :
    if goals[i] == '0':
        goals[i] = 0
    else:
        goals[i] = 1
# convert the data to numpy arrays
features = np.array(features)
goals = np.array(goals)
    
#convert these arrays to dataframes
features = pd.DataFrame(features)
goals = pd.DataFrame(goals)
# print(type(features))
# print(type(df.iloc[:, :-1]))
    
accuracy_list = []
#split data but not randomly

X_train, X_test, y_train, y_test = train_test_split(features, goals, test_size=0.2, random_state=98)
print(type(X_train))
# create the model

splitter = data[1]
max_depth = data[2]
criteria = data[0]
model = DecisionTreeClassifier(criterion=criteria, splitter=splitter, max_depth=max_depth, random_state=81)

# train the model
model.fit(X_train, y_train)

# make predictions
y_pred = model.predict(X_test)

# calculate the accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy: ", accuracy)
#print(max(accuracy_list))
# calculate the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix: \n", conf_matrix)

# calculate the classification report
class_report = classification_report(y_test, y_pred)
print("Classification Report: \n", class_report)

# visualise the tree
# import dtreeviz
# from dtreeviz.trees import dtreeviz

# viz = dtreeviz(model, X_train, y_train, target_name="goal", feature_names=df.columns[:-1], class_names=list(set(goals)))
# viz.save("decision_tree.svg")
# print("Decision Tree visualisation saved to 'decision_tree.svg'")

<class 'pandas.core.frame.DataFrame'>
Accuracy:  0.9016393442622951
Confusion Matrix: 
 [[38  2]
 [ 4 17]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.90      0.95      0.93        40
           1       0.89      0.81      0.85        21

    accuracy                           0.90        61
   macro avg       0.90      0.88      0.89        61
weighted avg       0.90      0.90      0.90        61



In [124]:
from dtreeviz import model as dtreeviz_model

import numpy as np


# Convert categorical features to numeric

X_train_np = np.array(X_train)
#print(y_train)
y_train_np = np.array(y_train)
# print((X_train_np))
for i in range(len(X_train_np)):
    for j in range(len(X_train_np[i])):
        print(X_train_np[i][j])
        X_train_np[i][j] = float(X_train_np[i][j])
new_y_train = []
for i in range(len(y_train_np)):
    new_y_train.append(int(y_train_np[i]))
y_train_np = np.array(new_y_train)


# print((X_train_np).shape)
# print((y_train_np))


# Create the dtreeviz visualization
viz = dtreeviz_model(
    model, 
    X_train_np, 
    y_train_np, 
    target_name="goal", 
    feature_names=list(df.columns[:-1]), 
    class_names=["No Disease", "Disease"]
)

# Save visualization
viz.view().save("decision_tree.svg")
print("Decision Tree visualization saved to 'decision_tree.svg'")


70.0
1.0
4.0
130.0
322.0
0.0
2.0
109.0
0.0
2.4
2.0
3.0
3.0
42.0
1.0
3.0
130.0
180.0
0.0
0.0
150.0
0.0
0.0
1.0
0.0
3.0
61.0
1.0
4.0
140.0
207.0
0.0
2.0
138.0
1.0
1.9
1.0
1.0
7.0
50.0
0.0
4.0
110.0
254.0
0.0
2.0
159.0
0.0
0.0
1.0
0.0
3.0
51.0
1.0
3.0
125.0
245.0
1.0
2.0
166.0
0.0
2.4
2.0
0.0
3.0
58.0
1.0
3.0
140.0
211.0
1.0
2.0
165.0
0.0
0.0
1.0
0.0
3.0
61.0
1.0
4.0
148.0
203.0
0.0
0.0
161.0
0.0
0.0
1.0
1.0
7.0
56.0
0.0
4.0
134.0
409.0
0.0
2.0
150.0
1.0
1.9
2.0
2.0
7.0
55.0
1.0
4.0
160.0
289.0
0.0
2.0
145.0
1.0
0.8
2.0
1.0
7.0
57.0
1.0
4.0
165.0
289.0
1.0
2.0
124.0
0.0
1.0
2.0
3.0
7.0
58.0
1.0
3.0
132.0
224.0
0.0
2.0
173.0
0.0
3.2
1.0
2.0
7.0
43.0
0.0
4.0
132.0
341.0
1.0
2.0
136.0
1.0
3.0
2.0
0.0
7.0
53.0
1.0
4.0
142.0
226.0
0.0
2.0
111.0
1.0
0.0
1.0
0.0
7.0
60.0
0.0
4.0
158.0
305.0
0.0
2.0
161.0
0.0
0.0
1.0
0.0
3.0
54.0
1.0
4.0
110.0
206.0
0.0
2.0
108.0
1.0
0.0
2.0
1.0
3.0
58.0
0.0
2.0
136.0
319.0
1.0
2.0
152.0
0.0
0.0
1.0
2.0
3.0
63.0
1.0
4.0
140.0
187.0
0.0
2.0
144.0
1.0
4.0
1.0
2.0
7

/tmp/ipykernel_2836/583141657.py:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
find

Decision Tree visualization saved to 'decision_tree.svg'
